# WavLM Utterance-Level Extraction (GPU)

Extracts per-utterance embeddings from audio segments.
Saves to Google Drive with checkpointing.

**~5 hours on T4 GPU for 645 videos.**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# Install
!pip install -q transformers librosa
import os, json, time
import numpy as np
import torch
import librosa
from tqdm import tqdm
from collections import defaultdict

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load utterance labels from Drive
UTT_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
print(f'Loading from {UTT_FILE}...')

utterances_by_video = defaultdict(list)
with open(UTT_FILE) as f:
    for line in f:
        u = json.loads(line)
        utterances_by_video[u['video_id']].append(u)

total_utts = sum(len(v) for v in utterances_by_video.values())
print(f'Loaded {total_utts} utterances from {len(utterances_by_video)} videos')

In [ ]:
# Find audio files on Drive
AUDIO_BASE = '/content/gdrive/MyDrive'
audio_paths = {}

# Check chuckle_audio
for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 'chuckle_audio_all/audio_all', 'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
    path = os.path.join(AUDIO_BASE, folder)
    if os.path.exists(path):
        for f in os.listdir(path):
            if f.endswith(('.mp3', '.wav')):
                vid = f.rsplit('.', 1)[0]
                if vid not in audio_paths:
                    audio_paths[vid] = os.path.join(path, f)

print(f'Found {len(audio_paths)} audio files')

In [ ]:
# Load model
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model on: {device}')

In [ ]:
# Setup output + checkpoint
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_utterance_embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'checkpoint.json')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'utterance_embeddings.jsonl')

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
    done_videos = set(checkpoint.get('done_videos', []))
    total_done = checkpoint.get('total_utterances', 0)
    print(f'Resuming: {len(done_videos)} videos done, {total_done} utterances')
else:
    done_videos = set()
    total_done = 0
    print('Starting fresh')

In [ ]:
# Extract utterance-level embeddings
SR = 16000
BATCH_SIZE = 32
CHECKPOINT_INTERVAL = 25

mode = 'a' if done_videos else 'w'
out_f = open(OUTPUT_FILE, mode)
t0 = time.time()
processed = 0
total_pos = 0

videos_to_process = [(vid, path) for vid, path in audio_paths.items() if vid not in done_videos and vid in utterances_by_video]
print(f'Processing {len(videos_to_process)} videos...')

for vid, audio_path in tqdm(videos_to_process):
    try:
        audio, sr = librosa.load(audio_path, sr=SR)
        
        # Collect all segments for this video
        segments = []
        utts = utterances_by_video[vid]
        for u in utts:
            start_sample = int(u['start'] * SR)
            end_sample = int(u['end'] * SR)
            if end_sample > len(audio):
                end_sample = len(audio)
            if start_sample >= len(audio) or end_sample - start_sample < SR * 0.02:
                continue
            segment = audio[start_sample:end_sample]
            if len(segment) < 400:
                segment = np.pad(segment, (0, 400 - len(segment)))
            segments.append((u, segment))
        
        # Batch process
        for batch_start in range(0, len(segments), BATCH_SIZE):
            batch = segments[batch_start:batch_start+BATCH_SIZE]
            batch_tensors = [torch.FloatTensor(s[1]) for s in batch]
            
            with torch.no_grad():
                for u, segment in batch:
                    inputs = torch.FloatTensor(segment).unsqueeze(0).to(device)
                    outputs = model(inputs)
                    emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                    
                    record = {
                        'uid': f"{vid}_{u['start']:.2f}",
                        'video_id': vid,
                        'start': u['start'],
                        'end': u['end'],
                        'text': u.get('text', '')[:200],
                        'label': u.get('label', 0),
                        'embedding': emb.tolist()
                    }
                    out_f.write(json.dumps(record) + '\n')
                    total_done += 1
                    if u.get('label', 0) == 1:
                        total_pos += 1
        
        done_videos.add(vid)
        processed += 1
        
    except Exception as e:
        print(f'Error {vid}: {e}')
        continue
    
    if processed % CHECKPOINT_INTERVAL == 0:
        out_f.flush()
        elapsed = time.time() - t0
        rate = processed / elapsed if elapsed > 0 else 0
        remaining = len(videos_to_process) - processed
        eta = remaining / rate / 3600 if rate > 0 else 0
        
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'done_videos': list(done_videos),
                'total_utterances': total_done,
                'total_positive': total_pos,
            }, f)
        
        print(f'\nCheckpoint: {processed}/{len(videos_to_process)} videos, {total_done} utts, ETA: {eta:.1f}h')

# Final save
out_f.close()
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump({
        'done_videos': list(done_videos),
        'total_utterances': total_done,
        'total_positive': total_pos,
        'completed': True,
    }, f, indent=2)

elapsed = time.time() - t0
print(f'\n{"="*60}')
print(f'DONE! {total_done} utterances, {total_pos} positive')
print(f'Time: {elapsed/3600:.1f} hours')
print(f'Output: {OUTPUT_FILE}')